In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib
import requests
import json
import pymysql

In [4]:
conn = pymysql.connect(
    host = '127.0.0.1',
    user = 'humanda6',
    passwd = 'humanda6',
    database = 'data_repo'
)
cur = conn.cursor()

def sql_get(sql):
    cur.execute(sql)
    return cur.fetchall()

def sql_set(sql):
    cur.execute(sql)
    conn.commit()


sql_set('''DROP TABLE IF EXISTS red_wine''')
sql_set("""create table red_wine
          (
            idx int primary key auto_increment,
            fixed_acidity float not null,
            volatile_acidity float not null,
            citric_acid float not null,
            residual_sugar float not null,
            chlorides float not null,
            free_sulfur_dioxide float not null,
            total_sulfur_dioxide float not null,
            density float not null,
            pH float not null,
            sulphates float not null,
            alcohol float not null,
            quality int not null
          )""")

df_redwine = pd.read_csv("data-files/winequality-red.csv", sep=';')
columns=['idx','fixed_acidity','volatile_acidity','citric_acid','residual_sugar','chlorides','free_sulfur_dioxide','total_sulfur_dioxide','density','pH','sulphates','alcohol','quality']

cur.executemany(f"""INSERT INTO red_wine ({','.join(columns[1:])}) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
                df_redwine.values.tolist() )
conn.commit()

df_iris = pd.DataFrame(sql_get('''SELECT * FROM red_wine'''), columns=columns)
df_iris.drop(['idx'], axis=1, inplace=True)
df_iris.to_csv('data-files/winequality-red2.csv', index=False)

cur.close()
conn.close()
